In [25]:
!pip install --upgrade kagglehub --quiet

import kagglehub

# Download latest version
path = kagglehub.dataset_download("jacksoncrow/stock-market-dataset")

print("Path to dataset files:", path)

!ls {path}

Using Colab cache for faster access to the 'stock-market-dataset' dataset.
Path to dataset files: /kaggle/input/stock-market-dataset
etfs  stocks  symbols_valid_meta.csv


In [26]:
!ls {path}/stocks/ | head

AACG.csv
AA.csv
AAL.csv
AAMC.csv
AAME.csv
AAN.csv
AAOI.csv
AAON.csv
AAP.csv
AAPL.csv


In [27]:
import pandas as pd

df = pd.read_csv(f"{path}/stocks/AACG.csv")
display(df.info(), df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       261 non-null    object 
 1   Open       261 non-null    float64
 2   High       261 non-null    float64
 3   Low        261 non-null    float64
 4   Close      261 non-null    float64
 5   Adj Close  261 non-null    float64
 6   Volume     261 non-null    int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 14.4+ KB


None

,Date,Open,High,Low,Close,Adj Close,Volume
0,2019-03-11,1.08,1.10,1.08,1.08,1.08,32100
1,2019-03-12,1.08,1.09,1.05,1.05,1.05,20200
2,2019-03-13,1.06,1.08,1.04,1.07,1.07,23100
3,2019-03-14,1.06,1.11,1.06,1.08,1.08,29900
4,2019-03-15,1.06,1.08,1.04,1.04,1.04,30900


In [28]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Dropout, Input

# Đảm bảo dữ liệu được sắp xếp theo thời gian
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# --- ĐIỂM THAY ĐỔI 1: TẠO NHÃN XU HƯỚNG ---
# Tính phần trăm thay đổi giá đóng cửa của ngày hôm sau so với ngày hôm nay
df['Next_Close'] = df['Close'].shift(-1)
df['Return'] = (df['Next_Close'] - df['Close']) / df['Close']

# Định nghĩa ngưỡng "Giữ nguyên" (ví dụ: thay đổi trong khoảng +/- 0.5%)
threshold = 0.005 

def classify_trend(r):
    if r > threshold:
        return 2  # Tăng
    elif r < -threshold:
        return 0  # Giảm
    else:
        return 1  # Giữ nguyên / Đi ngang

df['Trend'] = df['Return'].apply(classify_trend)

# Bỏ dòng cuối cùng vì không có dữ liệu của ngày hôm sau để tính toán xu hướng
df.dropna(inplace=True)

# 1. Tiền xử lý dữ liệu
features = ['Open', 'High', 'Low', 'Close', 'Volume']
dataset_features = df.filter(features).values
dataset_target = df['Trend'].values # Nhãn mục tiêu bây giờ là Trend

# Chuẩn hóa các đặc trưng đầu vào
feature_scaler = MinMaxScaler(feature_range=(0, 1))
scaled_features = feature_scaler.fit_transform(dataset_features)

# 2. Phân chia tập huấn luyện
training_data_len = int(np.ceil(len(scaled_features) * 0.8))

train_features = scaled_features[0:training_data_len, :]
train_target = dataset_target[0:training_data_len]

window_size = 30 
X_train = []
y_train = []

for i in range(window_size, len(train_features)):
    X_train.append(train_features[i-window_size:i, :])
    y_train.append(train_target[i]) # Lấy nhãn Trend

X_train, y_train = np.array(X_train), np.array(y_train)
print("Kích thước dữ liệu huấn luyện (X_train):", X_train.shape)

# 3. Xây dựng mô hình BiLSTM phân loại
tf.keras.backend.clear_session()

model = Sequential()
model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

model.add(Bidirectional(LSTM(units=50, return_sequences=True)))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(units=50, return_sequences=False)))
model.add(Dropout(0.2))

model.add(Dense(units=25, activation='relu'))

# --- ĐIỂM THAY ĐỔI 2 & 3: Lớp Dense cuối và Hàm compile ---
# Cần 3 units vì chúng ta có 3 lớp (0, 1, 2) và dùng hàm softmax để lấy xác suất
model.add(Dense(units=3, activation='softmax')) 

# Dùng sparse_categorical_crossentropy và theo dõi độ chính xác (accuracy)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Huấn luyện mô hình
print("Đang tiến hành huấn luyện mô hình phân loại xu hướng...")
model.fit(X_train, y_train, batch_size=16, epochs=20)

# 5. Chuẩn bị tập dữ liệu kiểm tra
test_features = scaled_features[training_data_len - window_size: , :]
X_test = []
y_test = dataset_target[training_data_len:] # Xu hướng thực tế

for i in range(window_size, len(test_features)):
    X_test.append(test_features[i-window_size:i, :])

X_test = np.array(X_test)

# 6. Đưa ra dự đoán
# Mô hình sẽ trả về ma trận xác suất, ta dùng argmax để lấy chỉ mục có xác suất cao nhất (0, 1 hoặc 2)
predicted_probs = model.predict(X_test)
predictions = np.argmax(predicted_probs, axis=1)

# 7. Đánh giá kết quả
print("\n--- KẾT QUẢ ĐÁNH GIÁ ---")
print(f"Độ chính xác chung (Accuracy): {accuracy_score(y_test, predictions) * 100:.2f}%")
print("\nBáo cáo chi tiết (0: Giảm, 1: Giữ nguyên, 2: Tăng):")
print(classification_report(y_test, predictions, zero_division=0))

Kích thước dữ liệu huấn luyện (X_train): (178, 30, 5)
Đang tiến hành huấn luyện mô hình phân loại xu hướng...
Epoch 1/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 87ms/step - accuracy: 0.3829 - loss: 1.0659
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.3940 - loss: 1.0135
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.4669 - loss: 0.9934
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.3891 - loss: 0.9974
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.4886 - loss: 0.9816
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.5344 - loss: 0.9399
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.5403 - loss: 0.9699
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.4434 - loss: 0.9529
Epoch 9/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.4041 - loss: 1.0519
Epoch 10/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.4132 - loss: 1.0049
Epoch 11/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 